# Date Attribution and Dissolve Workflow for EDO Vector Outputs

This notebook prepares drought-related vector datasets exported from ArcGIS processing workflows.

The workflow:
- connects to a file geodatabase,
- lists available feature classes,
- selects feature classes containing `final` in their name,
- extracts acquisition dates from feature class names,
- writes the extracted date into a new attribute field,
- exports the datasets as shapefiles,
- and dissolves the exported shapefiles by `gridcode` and `date`.

The notebook is part of a GIS thesis workflow for drought analysis in Hungary.

## ArcPy Environment Requirement

This notebook requires **ArcPy**, therefore it must be run with an **ArcGIS Pro Python kernel**.

Before running the notebook:
- switch the Jupyter kernel to the ArcGIS Pro environment, for example `arcgispro-py3`,
- or open and run the notebook directly from ArcGIS Pro.

A standard Anaconda or Jupyter environment will not run this notebook unless ArcPy is available.

## Import Libraries and Define Paths

This section imports the required Python modules and defines the input geodatabase and output folder.

The paths below are written as relative paths for portability. They should be adjusted if the local project folder structure is different.

In [ ]:
import arcpy
import os
import re
import glob
from datetime import datetime

# --- Paths ---
gdb = r"./Results_copy.gdb"
out_folder = r"./Data/EuropeanDroughtObservatory/Data_dated_py"

# Create output folder if it does not exist
os.makedirs(out_folder, exist_ok=True)

# ArcPy workspace settings
arcpy.env.workspace = gdb
arcpy.env.overwriteOutput = True

## List Feature Classes in the Geodatabase

This step lists all feature classes available in the selected geodatabase.

It is used as a quick check to confirm that the correct workspace has been selected before processing.

In [ ]:
fcs = arcpy.ListFeatureClasses()

print("Total feature classes:", len(fcs))

for fc in fcs:
    print(fc)

## Select Final Feature Classes

Only feature classes containing `final` in their name are selected for further processing.

This prevents intermediate or unrelated geodatabase layers from being processed accidentally.

In [ ]:
final_fcs = [fc for fc in fcs if "final" in fc.lower()]

print("Final feature classes selected:", len(final_fcs))

for fc in final_fcs:
    print(fc)

## Extract Dates and Export Feature Classes

For each selected feature class, the script searches for an eight-digit date pattern (`YYYYMMDD`) in the feature class name.

If a date is found:
- a `date` field is added if it does not already exist,
- the extracted date is written into the field,
- and the feature class is exported as a shapefile.

The exported shapefiles are saved into the output folder.

In [ ]:
for fc in final_fcs:

    print("Processing:", fc)

    # Search for YYYYMMDD date pattern in feature class name
    match = re.search(r"(\d{8})", fc)
    if not match:
        print("  -> No date found, skipped.")
        continue

    date_str = match.group(1)
    date_obj = datetime.strptime(date_str, "%Y%m%d")

    # Add date field if missing
    fields = [f.name for f in arcpy.ListFields(fc)]

    if "date" not in fields:
        arcpy.AddField_management(fc, "date", "DATE")

    # Calculate date field
    expr = f"datetime({date_obj.year}, {date_obj.month}, {date_obj.day})"

    arcpy.CalculateField_management(
        fc,
        "date",
        expr,
        "PYTHON3"
    )

    # Export to shapefile
    out_name = f"EDO_Hun_{date_str}.shp"
    out_path = os.path.join(out_folder, out_name)

    arcpy.CopyFeatures_management(fc, out_path)

    print("  -> export ready:", out_name)

print("Files ready.")

## Find Exported Shapefiles

This step collects all exported shapefiles from the output folder.

These files will be used as input for the dissolve operation.

In [ ]:
shps = glob.glob(os.path.join(out_folder, "EDO_Hun_*.shp"))

print("Exported shapefiles found:", len(shps))

for shp in shps:
    print(shp)

## Dissolve Shapefiles by Grid Code and Date

The exported shapefiles are dissolved by `gridcode` and `date`.

This reduces internal boundaries between neighbouring polygons that share the same drought category and date, producing cleaner outputs for later analysis or visualization.

In [ ]:
for shp in shps:

    print("Dissolve:", shp)

    out_name = shp.replace(".shp", "_diss.shp")
    out_path = os.path.join(out_folder, out_name)

    arcpy.Dissolve_management(
        shp,
        out_path,
        ["gridcode", "date"]
    )

    print("  -> ready:", out_name)

print("Dissolve done.")